# colab_09 — Geneformer zero-shot embedding (regime #1 baseline)

First notebook in the **foundation-model (FM) work**. It produces the **zero-shot** Geneformer embedding of the glia substrate: the un-adapted pretrained model's per-cell vectors, no fine-tuning. Zero-shot is regime #1 of the evaluation contract and does double duty — a competency deliverable in its own right ("embed with an FM, inspect the space critically") **and** the frozen baseline that detector #1 (embedding drift) and evals #1–#2 measure every later CPT run against. Getting it right and reproducible matters for everything downstream.

Geneformer's `setup.py` requires only Python ≥3.10 and does **not** depend on scvi-tools or scgpt, so this notebook runs on Colab's **native Python** — no runtime downgrade. scGPT, which does pin an older <3.11 stack, gets its own environment in colab_10 (`requirements_scgpt.txt`); the two FMs are therefore installed in separate envs (Option-B env split, 2026-07-03) rather than one shared FM env.

The pipeline: (§1–§2) stand up + capture the Geneformer environment — where the deliberately-`TBD` pins in `requirements_geneformer.txt` get resolved; (§3) load the two label-carrying subset files (`micro_subset.h5ad` from colab_07, `astro_subset.h5ad` from colab_08 — raw counts, full ca. 26,514-gene panel, `substate` + `apoe_carrier` attached) and combine them; (§4) **vocab audit** — Geneformer tokenizes by Ensembl ID over a fixed vocabulary, so map our gene symbols across and confirm the niche-critical genes survive, with **APOE out-of-vocab as a pre-registered hard fail** (it would make eval #2 impossible for this FM); (§5) tokenize + embed; (§6) interrogate the raw embedding space; (§7) save the baseline + audit trace.

Runs on a **GPU** runtime (the forward pass through the pretrained model); high-RAM helpful for the combined ca. 142k-cell object. No training happens here — that is colab_11+ (CPT).

## 1 — Setup


### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment

Same Drive + repo bootstrap as the integration notebooks, run on Colab's native Python (no runtime swap — Geneformer needs only ≥3.10). Installs `requirements_geneformer.txt` (the lean Geneformer-only stack) and clones **Geneformer V2** from its Hugging Face repo (it is not on PyPI); `pip install .` on the clone pulls Geneformer's own deps (transformers / torch / peft / datasets / scanpy / anndata / …). scGPT is deliberately **not** installed here — it has conflicting older pins and lives in its own colab_10 environment (`requirements_scgpt.txt`).

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Geneformer requires Python >=3.10 (setup.py) and runs on Colab's native interpreter --
# no downgrade. Fail loud only if Colab ever ships something older than Geneformer allows.
assert sys.version_info >= (3, 10), (
    f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."
)

# Lean Geneformer-only stack: rides Colab's native numpy-2 base (NO numpy/scipy pin --
# see the NUMPY POLICY note in requirements_geneformer.txt); scGPT is NOT
# installed here (separate colab_10 env). Geneformer's own deps come from `pip install .` below.
!pip install -r {REPO_PATH}/requirements_geneformer.txt

# Geneformer V2 -- installed from the HF repo, NOT pypi (requirements_geneformer.txt note).
GENEFORMER_REPO = "/content/Geneformer"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && pip install .

# Record the exact Geneformer commit so vocab + model weights are reproducible (pin after run).
GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True).stdout.strip()
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT)

## 2 — Environment capture


### 2a — pip freeze + env JSON (resolves the `TBD` Geneformer pins)

`requirements_geneformer.txt` intentionally leaves the heavy stack (torch / transformers / peft / accelerate / datasets) unpinned, to be resolved at the first real install by Geneformer's own `pip install .`. This cell freezes the resolved environment and records the Geneformer commit, GPU, and CUDA. After a clean run, the concrete versions are copied back into `requirements_geneformer.txt` and the pins locked. (`scgpt_version` is recorded too and will read `None` here — scGPT is not in this env by design.)

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_09"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

# The heavy pins in requirements_geneformer.txt were left TBD by design (resolved by
# Geneformer's `pip install .`) — this freeze resolves them. After a clean run, copy the
# concrete versions below back into requirements_geneformer.txt and pin the Geneformer commit.
env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "geneformer_commit":    GENEFORMER_COMMIT,
    "scanpy_version":       _ver("scanpy"),
    "anndata_version":      _ver("anndata"),
    "torch_version":        _ver("torch"),
    "transformers_version": _ver("transformers"),
    "peft_version":         _ver("peft"),
    "accelerate_version":   _ver("accelerate"),
    "datasets_version":     _ver("datasets"),
    "scgpt_version":        _ver("scgpt"),   # None in this env by design (scGPT is colab_10)
    "geneformer_version":   _ver("geneformer"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Load the glia substrate


### 3a — Load both labelled subsets, combine, and guard raw counts

Reads `micro_subset.h5ad` (colab_07) and `astro_subset.h5ad` (colab_08) — the post-QC-drop, substate-labelled halves of the glia set — and concatenates them into one ca. 142k-cell object for a single tokenize/embed pass. Both descend from the same colab_05 glia object, so the gene panel is asserted identical before concatenation. A `cell_index` is stamped on every cell as a stable key to realign embeddings after tokenizing (Geneformer's output is not guaranteed to preserve row order). The raw-counts guard is not optional here: Geneformer rank-encodes **raw** counts, so a log-normalized `.X` would produce a silently wrong embedding.


In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1
FIG_DIR = os.path.join(REPO_PATH, "figures", "colab_09")
os.makedirs(FIG_DIR, exist_ok=True)

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape, "| substate:", micro.obs["substate"].value_counts().to_dict())
print("astrocyte subset:", astro.shape, "| substate:", astro.obs["substate"].value_counts().to_dict())

# Both subsets descend from the same colab_05 glia object -> identical gene panel (full
# ca. 26,514-gene intersection, raw counts in .X). Assert before concatenating; a mismatch
# would mean one file was regenerated against a different intersection.
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

# combine into one glia object for a single tokenize/embed pass; keep lineage explicit.
micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)   # stable key to realign embeddings after tokenizing
print("\ncombined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("apoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())

# raw-counts guard — Geneformer rank-encodes RAW counts; log-normalized input is silently wrong.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}) — FM input must be raw"
print("raw-counts int-frac:", round(frac_int, 3))
_ram("combined glia")


## 4 — Vocabulary audit (setup audit A)


### 4a — Map gene symbols to Ensembl, intersect the Geneformer vocabulary, gate on APOE

Geneformer does not read gene symbols — it tokenizes by Ensembl ID over a vocabulary fixed at its pretraining, and any measured gene outside that vocabulary is dropped from the cell's input. This cell quantifies that loss: how many of our ca. 26,514 genes map to an Ensembl ID and how many of those are in-vocabulary, then checks the niche-critical genes one by one (APOE / TREM2 / MS4A6A / CLU / GFAP / AQP4 / AIF1 / CSF1R). **APOE out-of-vocab is a pre-registered hard fail** — without APOE tokenized, eval #2 (the Stanton-core APOE axis) cannot be run for Geneformer, so the notebook stops rather than produce an embedding that silently cannot answer the load-bearing question. The other niche genes are warn-only, logged to the audit report.


In [ ]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
import pickle

# geneformer/__init__.py exports these as absolute Path objects pointing at the GC104M
# dictionaries shipped inside the installed package — the same dictionaries the tokenizer
# (§5a) uses by default and that back the Geneformer-V2-104M checkpoint used in §5b.
# Confirmed against the Geneformer source (2026-07-02) — no globbing/guessing needed.
GENE_NAME_ID_FILE = ENSEMBL_DICTIONARY_FILE   # gene SYMBOL -> Ensembl ID
TOKEN_DICT_FILE   = TOKEN_DICTIONARY_FILE     # Ensembl ID -> integer token
print("using symbol->ensembl:", GENE_NAME_ID_FILE.name)
print("using token dict:     ", TOKEN_DICT_FILE.name)

with open(GENE_NAME_ID_FILE, "rb") as f:
    symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICT_FILE, "rb") as f:
    token_dictionary = pickle.load(f)   # keys = Ensembl IDs Geneformer can tokenize

# Map our gene symbols -> Ensembl, then intersect with the token vocabulary.
glia.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in glia.var_names]
mapped   = glia.var["ensembl_id"].notna()
in_vocab = glia.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)

n_total  = glia.n_vars
n_mapped = int(mapped.sum())
n_vocab  = int(in_vocab.sum())
print(f"\ngene panel: {n_total}")
print(f"  symbol->ensembl mapped : {n_mapped}  ({n_mapped/n_total:.1%})")
print(f"  in Geneformer vocab    : {n_vocab}  ({n_vocab/n_total:.1%})")

# Niche-critical survival — the genes the whole niche rests on must be tokenizable.
NICHE_CRITICAL_GENES = ["APOE", "TREM2", "MS4A6A", "CLU", "GFAP", "AQP4", "AIF1", "CSF1R"]
panel = set(glia.var_names)
niche_status = {}
for g in NICHE_CRITICAL_GENES:
    e = symbol_to_ensembl.get(g) if g in panel else None
    niche_status[g] = {
        "in_panel": g in panel,
        "ensembl":  e,
        "in_vocab": bool(e in token_dictionary) if e is not None else False,
    }
print("\nniche-critical gene survival:")
for g, s in niche_status.items():
    flag = "OK" if s["in_vocab"] else ("WARN" if s["in_panel"] else "ABSENT")
    print(f"  {g:8s} panel={str(s['in_panel']):5} vocab={str(s['in_vocab']):5}  [{flag}]")

# Pre-registered gate: APOE out-of-vocab makes eval #2 (Stanton-core APOE axis) impossible
# for Geneformer -> HARD FAIL. The other niche genes are warn-only (logged, not fatal).
assert niche_status["APOE"]["in_vocab"], (
    "APOE is not in the Geneformer vocabulary — eval #2 cannot be run for this FM. "
    "Pre-registered hard fail (EVALUATION_CONTRACT eval #2)."
)
niche_warnings = [g for g, s in niche_status.items() if not s["in_vocab"]]
if niche_warnings:
    print("\nWARN — niche genes not tokenizable (logged to audit):", niche_warnings)

VOCAB_AUDIT = {
    "n_genes_panel":    n_total,
    "n_mapped_ensembl": n_mapped,
    "n_in_vocab":       n_vocab,
    "frac_in_vocab":    round(n_vocab / n_total, 4),
    "niche_status":     niche_status,
    "niche_warnings":   niche_warnings,
    "apoe_hard_fail_gate": "passed",
}

## 5 — Tokenize + embed


### 5a — Tokenize the glia object into a Geneformer dataset

Geneformer's tokenizer converts each cell's raw expression into a rank-ordered sequence of gene tokens (highest-expressed genes first, normalized by each cell's total counts). It reads h5ad files from a directory, requiring `var["ensembl_id"]` (added in §4a) and `obs["n_counts"]` (total raw counts). The chosen label columns are carried through into the tokenized dataset so every embedded cell keeps its lineage / substate / APOE / study / donor labels and its `cell_index`.


In [ ]:
from geneformer import TranscriptomeTokenizer

# Geneformer normalizes each cell by n_counts; recompute from raw .X (fail loud on zero-count cells).
glia.obs["n_counts"] = np.asarray(glia.X.sum(axis=1)).ravel()
assert (glia.obs["n_counts"] > 0).all(), "cells with zero counts present — should have been QC'd upstream"

TOK_IN_DIR  = "/content/gf_token_in"
TOK_OUT_DIR = "/content/gf_token_out"
os.makedirs(TOK_IN_DIR, exist_ok=True)
os.makedirs(TOK_OUT_DIR, exist_ok=True)
glia.write_h5ad(os.path.join(TOK_IN_DIR, "glia.h5ad"))

# custom_attr_name_dict: obs column -> name carried through into the .dataset, so each
# tokenized cell keeps its labels + the stable cell_index for realignment after embedding.
CUSTOM_ATTRS = {
    "cell_index":   "cell_index",
    "lineage":      "lineage",
    "substate":     "substate",
    "apoe_carrier": "apoe_carrier",
    "study_id":     "study_id",
    "donor_id":     "donor_id",
}
# Signature confirmed against Geneformer source (2026-07-02): custom_attr_name_dict is the
# first positional arg; defaults (model_version="V2", model_input_size=4096, special_token=True,
# and the GC104M gene_median/token_dictionary/gene_mapping files) already match the
# Geneformer-V2-104M checkpoint used in §5b, so nothing else needs to be passed here.
tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, "glia_zeroshot", file_format="h5ad")
TOKENIZED_DATASET = os.path.join(TOK_OUT_DIR, "glia_zeroshot.dataset")
print("tokenized ->", TOKENIZED_DATASET)

### 5b — Extract zero-shot cell embeddings from the pretrained model

Runs the forward pass through the **un-adapted** pretrained Geneformer checkpoint and pools each cell's final-layer token representations into one cell-embedding vector — the zero-shot representation. The result is realigned to the combined object by `cell_index` (the tokenizer/extractor does not guarantee row order) and stored in `obsm["X_geneformer"]`.


In [ ]:
from geneformer import EmbExtractor

# Zero-shot cell embeddings from the un-adapted pretrained model (regime #1 = baseline).
# Confirmed against the repo tree (2026-07-02): top-level checkpoint dirs are
# Geneformer-V1-10M / Geneformer-V2-104M / Geneformer-V2-104M_CLcancer / Geneformer-V2-316M.
# 104M is the standard V2 pretrained checkpoint (matches the GC104M dictionaries used in §4a).
MODEL_DIR = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
assert os.path.exists(MODEL_DIR), f"model dir not found: {MODEL_DIR} — check repo layout"

# emb_mode="cell" and emb_label are both confirmed valid EmbExtractor options (source,
# 2026-07-02): valid_option_dict restricts emb_mode to {"cls","cell","gene"}, and
# max_ncells=None is explicitly documented to mean "all cells".
ee = EmbExtractor(
    model_type="Pretrained",
    num_classes=0,
    emb_mode="cell",
    max_ncells=None,                # embed every cell
    emb_layer=-1,                   # final hidden layer
    emb_label=["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"],
    forward_batch_size=64,          # tune to GPU memory
    nproc=4,
)
EMB_OUT_DIR = "/content/gf_emb_out"
os.makedirs(EMB_OUT_DIR, exist_ok=True)
emb_df = ee.extract_embs(MODEL_DIR, TOKENIZED_DATASET, EMB_OUT_DIR, "glia_zeroshot")
print("embedding frame:", emb_df.shape)

# extract_embs returns embedding-dim columns followed by the emb_label columns, row order
# preserved from the input dataset (confirmed against source, 2026-07-02) — the explicit
# cell_index realignment below is kept anyway rather than trusted-by-default, in case a
# future Geneformer version changes that guarantee.
label_cols = ["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
emb_cols = [c for c in emb_df.columns if c not in label_cols]
emb_df = emb_df.set_index("cell_index").reindex(glia.obs["cell_index"].values)
assert emb_df[emb_cols].notna().all().all(), "embedding rows missing after realignment to cell_index"
X_gf = emb_df[emb_cols].to_numpy(dtype=np.float32)
glia.obsm["X_geneformer"] = X_gf
print("X_geneformer:", X_gf.shape, "| emb dim:", X_gf.shape[1])

## 6 — Interrogate the zero-shot space


### 6a — UMAP + silhouettes: what does the raw FM embedding actually encode?

The critical-inspection step — the direct answer to the postdoc's "we don't know what vector space it is." A UMAP of the zero-shot embedding coloured by lineage, study, APOE-carrier, and substate, plus silhouette scores, ask three questions before any fine-tuning: does astro-vs-microglia **lineage** identity saturate the space (expected — any embedding separates the lineages); is **study** batch structure visible that the FM did not correct (FMs are claimed batch-robust — this tests it directly); and is there any **APOE** geometry within each lineage at all (colab_06 found no clean E4 structure in the scVI latent — this previews whether eval #2 has anything to recover even before CPT). A zero-shot embedding that already separates substate, or shows heavy study batching, changes how every later CPT result is read.


In [ ]:
from sklearn.metrics import silhouette_score

sc.pp.neighbors(glia, use_rep="X_geneformer", n_neighbors=15)
sc.tl.umap(glia)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, key in zip(axes.ravel(), ["lineage", "study_id", "apoe_carrier", "substate"]):
    sc.pl.umap(glia, color=key, ax=ax, show=False, frameon=False, size=4,
               title=f"Geneformer zero-shot — {key}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "6a_geneformer_zeroshot_umap.png"), dpi=150, bbox_inches="tight")
plt.show()

# Quantify what the raw FM space encodes BEFORE any adaptation.
# apoe_carrier is not binary as stored: EVALUATION_CONTRACT locks E4-carrier vs non-carrier,
# with E2-without-E4 cells EXCLUDED from the contrast (EXCLUDE_VALS = {"e2"}, colab_06 §6b) —
# not scored as a third class. The generic _sil helper only drops NaN, so an apoe_label call
# must pre-filter "e2" explicitly or the silhouette silently mixes in a third, off-contract group.
APOE_EXCLUDE_VALS = {"e2"}

def _sil(mask, label, exclude_vals=None):
    sub = glia[mask].copy()
    keep = sub.obs[label].notna().values
    if exclude_vals:
        keep &= ~sub.obs[label].astype(str).isin(exclude_vals).values
    sub = sub[keep]
    y = sub.obs[label].astype(str).values
    if len(set(y)) < 2 or sub.n_obs < 50:
        return None
    n = min(10000, sub.n_obs)
    idx = np.random.default_rng(0).choice(sub.n_obs, n, replace=False)
    return round(float(silhouette_score(sub.obsm["X_geneformer"][idx], y[idx])), 4)

all_mask   = np.ones(glia.n_obs, bool)
micro_mask = (glia.obs["lineage"] == "microglia").values
astro_mask = (glia.obs["lineage"] == "astrocyte").values
sil = {
    "lineage_all":    _sil(all_mask,   "lineage"),   # expect HIGH — identity saturates
    "study_all":      _sil(all_mask,   "study_id"),  # batch the FM did NOT correct
    "apoe_micro":     _sil(micro_mask, "apoe_carrier", exclude_vals=APOE_EXCLUDE_VALS),
    "apoe_astro":     _sil(astro_mask, "apoe_carrier", exclude_vals=APOE_EXCLUDE_VALS),
    "substate_micro": _sil(micro_mask, "substate"),
    "substate_astro": _sil(astro_mask, "substate"),
}
print("silhouettes (zero-shot Geneformer):")
for k, v in sil.items():
    print(f"  {k:16s}: {v}")


## 7 — Save + handoff


### 7a — Save the zero-shot baseline embedding, append audit trace, print commit commands

Writes a compact baseline artifact — per-cell embedding vectors plus the labels the evals slice on (no gene matrix, so it stays small) — to Drive as `glia_geneformer_zeroshot.h5ad`. This is the frozen reference every later CPT embedding is compared against (detector #1 drift; evals #1/#2), aligned by `cell_index`. The vocab-audit result and zero-shot silhouettes are appended to `outputs/audit_report.json`, and the WSL commit/push commands are printed (Colab has no git credentials).


In [ ]:
import shlex

GF_DIR = os.path.join(DRIVE_ROOT, "geneformer")
os.makedirs(GF_DIR, exist_ok=True)

# Lean baseline artifact: per-cell zero-shot vectors + labels. detector #1 and evals #1/#2
# compare post-CPT embeddings against THIS, aligned by cell_index — keep it compact (no gene
# matrix) but keep every label the evals slice on.
emb_adata = ad.AnnData(
    X=glia.obsm["X_geneformer"],
    obs=glia.obs[["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]].copy(),
)
emb_adata.obsm["X_umap"] = glia.obsm["X_umap"]
EMB_PATH = os.path.join(GF_DIR, "glia_geneformer_zeroshot.h5ad")
emb_adata.write_h5ad(EMB_PATH)
print("saved zero-shot embedding ->", EMB_PATH, f"({os.path.getsize(EMB_PATH)/1e9:.2f} GB)")

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)
report["geneformer_zeroshot"] = {
    "status": "computed",
    "date": TODAY,
    "model_dir": os.path.basename(MODEL_DIR),
    "geneformer_commit": GENEFORMER_COMMIT,
    "n_cells": int(glia.n_obs),
    "emb_dim": int(glia.obsm["X_geneformer"].shape[1]),
    "vocab_audit": VOCAB_AUDIT,
    "zeroshot_silhouettes": sil,
    "embedding_file": os.path.relpath(EMB_PATH, DRIVE_ROOT),
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL — Colab has no git creds) ===")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
      "'colab_09: Geneformer zero-shot embedding + vocab audit (regime #1 baseline)'")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")


### Carried forward

- **`glia_geneformer_zeroshot.h5ad`** (Drive) — the regime-#1 baseline embedding; colab_11+ (CPT) compares post-fine-tuning embeddings against it for detector #1 and evals #1–#2, aligned by `cell_index`.
- **Resolved Geneformer-env pins** (`outputs/software_versions/colab_09_*`) — copy the concrete versions back into `requirements_geneformer.txt` and lock, plus the Geneformer commit.
- **colab_10** repeats this skeleton for **scGPT** (gene-symbol -> vocab -> binned expression) in its own `requirements_scgpt.txt` environment (Python 3.10, the older scvi-tools/scgpt stack), producing the parallel zero-shot baseline; the two FMs are always reported together.